In [4]:
!git clone https://github.com/dianahamidullina14/SMILES-2026-ZO-Limited-Resnet.git
%cd SMILES-2026-ZO-Limited-Resnet

Cloning into 'SMILES-2026-ZO-Limited-Resnet'...
remote: Enumerating objects: 28, done.
remote: Counting objects: 100% (28/28), done.
remote: Compressing objects: 100% (20/20), done.
remote: Total 28 (delta 12), reused 21 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (28/28), 18.41 KiB | 6.14 MiB/s, done.
Resolving deltas: 100% (12/12), done.
/kaggle/working/SMILES-2026-ZO-Limited-Resnet


In [5]:
!pip install -r requirements.txt -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.5/78.5 kB 5.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.


In [10]:
import os
os.chdir('/kaggle/working/SMILES-2026-ZO-Limited-Resnet')

In [11]:
code = '''import pickle
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from augmentation import get_transforms


class CIFAR100Dataset(Dataset):
    def __init__(self, data_path, transform=None):
        with open(data_path, "rb") as f:
            d = pickle.load(f, encoding="bytes")
        self.images = d[b"data"].reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)
        self.labels = d[b"fine_labels"]
        self.transform = transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        from PIL import Image
        img = Image.fromarray(self.images[idx])
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]


def get_train_dataset_loader(data_dir, batch_size, generator_train=None):
    transform = get_transforms(train=True)
    dataset = CIFAR100Dataset(
        "/kaggle/input/datasets/fedesoriano/cifar100/train",
        transform=transform
    )
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=2,
        pin_memory=True,
        generator=generator_train,
    )
    return dataset, loader
'''
with open('train_data.py', 'w') as f:
    f.write(code)
print("train_data.py OK")

train_data.py OK


In [12]:
code = '''import torch.nn as nn


def init_last_layer(layer: nn.Linear) -> None:
    nn.init.xavier_uniform_(layer.weight)
    nn.init.zeros_(layer.bias)
'''
with open('head_init.py', 'w') as f:
    f.write(code)
print("head_init.py OK")

head_init.py OK


In [13]:
code = '''import torchvision.transforms as T

_CIFAR100_MEAN = (0.5071, 0.4867, 0.4408)
_CIFAR100_STD  = (0.2675, 0.2565, 0.2761)


def get_transforms(train: bool) -> T.Compose:
    if train:
        return T.Compose([
            T.Resize(224),
            T.RandomCrop(224, padding=28),
            T.RandomHorizontalFlip(),
            T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
            T.ToTensor(),
            T.Normalize(mean=_CIFAR100_MEAN, std=_CIFAR100_STD),
            T.RandomErasing(p=0.2),
        ])
    else:
        return T.Compose([
            T.Resize(224),
            T.ToTensor(),
            T.Normalize(mean=_CIFAR100_MEAN, std=_CIFAR100_STD),
        ])
'''
with open('augmentation.py', 'w') as f:
    f.write(code)
print("augmentation.py OK")

augmentation.py OK


In [14]:
code = '''from __future__ import annotations
from typing import Callable
import torch
import torch.nn as nn


class ZeroOrderOptimizer:
    def __init__(
        self,
        model: nn.Module,
        lr: float = 0.05,
        eps: float = 0.001,
        perturbation_mode: str = "gaussian",
    ) -> None:
        self.model = model
        self.lr = lr
        self.eps = eps
        self.perturbation_mode = perturbation_mode
        self.step_count = 0

        # Тюним только голову — самый эффективный выбор при малом бюджете
        self.layer_names: list[str] = ["fc.weight", "fc.bias"]

        # Adam-моменты для более умного обновления
        self._m: dict[str, torch.Tensor] = {}
        self._v: dict[str, torch.Tensor] = {}
        self._beta1 = 0.9
        self._beta2 = 0.999
        self._adam_eps = 1e-8

    def _active_params(self) -> dict[str, nn.Parameter]:
        named = dict(self.model.named_parameters())
        missing = [n for n in self.layer_names if n not in named]
        if missing:
            raise KeyError(f"Layer names not found: {missing}")
        return {n: named[n] for n in self.layer_names}

    def _estimate_grad(
        self,
        loss_fn: Callable[[], float],
        params: dict[str, nn.Parameter],
    ) -> dict[str, torch.Tensor]:
        # SPSA: один общий вектор возмущения для всех параметров
        # Всего 2 вызова модели на весь шаг — не зависит от числа параметров
        perturbations = {}
        with torch.no_grad():
            for name, param in params.items():
                # Rademacher: случайные {-1, +1}
                delta = torch.randint(0, 2, param.shape,
                                      device=param.device).float() * 2 - 1
                perturbations[name] = delta

            # f(x + eps * delta)
            for name, param in params.items():
                param.data.add_(self.eps * perturbations[name])
            f_plus = loss_fn()

            # f(x - 2*eps * delta) = f(x - eps*delta) после прибавления
            for name, param in params.items():
                param.data.sub_(2.0 * self.eps * perturbations[name])
            f_minus = loss_fn()

            # Восстанавливаем
            for name, param in params.items():
                param.data.add_(self.eps * perturbations[name])

        grad_scalar = (f_plus - f_minus) / (2.0 * self.eps)
        grads = {}
        for name in params:
            grads[name] = grad_scalar * perturbations[name]

        return grads

    def _update_params(
        self,
        params: dict[str, nn.Parameter],
        grads: dict[str, torch.Tensor],
    ) -> None:
        self.step_count += 1
        t = self.step_count
        bc1 = 1.0 - self._beta1 ** t
        bc2 = 1.0 - self._beta2 ** t

        with torch.no_grad():
            for name, param in params.items():
                g = grads[name]

                if name not in self._m:
                    self._m[name] = torch.zeros_like(param)
                    self._v[name] = torch.zeros_like(param)

                # Adam update
                self._m[name] = self._beta1 * self._m[name] + (1 - self._beta1) * g
                self._v[name] = self._beta2 * self._v[name] + (1 - self._beta2) * g * g

                m_hat = self._m[name] / bc1
                v_hat = self._v[name] / bc2

                param.data.sub_(self.lr * m_hat / (v_hat.sqrt() + self._adam_eps))

    def step(self, loss_fn: Callable[[], float]) -> float:
        params = self._active_params()
        with torch.no_grad():
            loss_before = loss_fn()
        grads = self._estimate_grad(loss_fn, params)
        self._update_params(params, grads)
        return float(loss_before)
'''
with open('zo_optimizer.py', 'w') as f:
    f.write(code)
print("zo_optimizer.py OK")

zo_optimizer.py OK


### validate.py пытается скачать val-сет — у меня это падает с 503, поэтому я подменяю его через symlink на kaggle-данные

In [15]:
import os, subprocess

val_src = '/kaggle/input/datasets/fedesoriano/cifar100'
data_dir = '/kaggle/working/data'
os.makedirs(data_dir, exist_ok=True)

# Создаём структуру которую ожидает torchvision
cifar_dir = os.path.join(data_dir, 'cifar-100-python')
os.makedirs(cifar_dir, exist_ok=True)

import shutil
for fname in ['train', 'test', 'meta']:
    src = os.path.join(val_src, fname)
    dst = os.path.join(cifar_dir, fname)
    if not os.path.exists(dst):
        shutil.copy(src, dst)
        print(f"Copied {fname}")

print("Data dir ready:", os.listdir(cifar_dir))

Copied train
Copied test
Copied meta
Data dir ready: ['test', 'meta', 'train']


In [16]:
!python validate.py --data_dir /kaggle/working/data --batch_size 32 --n_batches 256 --output results.json

[Device] Using: cuda
[Data] Loading CIFAR100 from '/kaggle/working/data' ...
[Data] Train: 50,000 samples | Val: 10,000 samples

[Checkpoint 1/3] Baseline (ImageNet head)
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|███████████████████████████████████████| 44.7M/44.7M [00:00<00:00, 140MB/s]
  Top-1: 0.37%                                                                  

[Checkpoint 2/3] Initialized head (no fine-tuning)
  Top-1: 1.21%                                                                  

[Checkpoint 3/3] ZO fine-tuning (256 steps)
  Active layers: ['fc.weight', 'fc.bias']
  Fine-tuning: 100%|██████████| 256/256 [00:23<00:00, 11.11step/s, loss=31.4104]
  Evaluating fine-tuned model ...
  Top-1: 1.20%                                                                  

 Evaluation Summary
  Checkpoint                        Top-1
-------------------------------------------------------

# Бейзлайн дает очень низкое качетсво, поэтому посмортрим что происхоидт с loss 

In [17]:
import torch
import sys
sys.path.insert(0, '/kaggle/working/SMILES-2026-ZO-Limited-Resnet/SMILES-2026-ZO-Limited-Resnet')

from model import get_model
from train_data import get_train_dataset_loader
from zo_optimizer import ZeroOrderOptimizer
import torch.nn as nn

device = torch.device('cuda')
model = get_model().to(device)
optimizer = ZeroOrderOptimizer(model)
train_dataset, train_loader = get_train_dataset_loader(
    data_dir='/kaggle/working/data',
    batch_size=64,
    generator_train=None
)
criterion = nn.CrossEntropyLoss()

def infinite(loader):
    while True:
        yield from loader

data_iter = infinite(train_loader)
losses = []

for i in range(30):
    images, labels = next(data_iter)
    images, labels = images.to(device), labels.to(device)
    
    def loss_fn(img=images, lbl=labels):
        model.eval()
        with torch.no_grad():
            return float(criterion(model(img), lbl).item())
    
    loss = optimizer.step(loss_fn)
    losses.append(loss)
    if i % 5 == 0:
        print(f"Step {i:3d} | loss={loss:.4f}")

print(f"\nFirst 5 losses: {[f'{x:.3f}' for x in losses[:5]]}")
print(f"Last  5 losses: {[f'{x:.3f}' for x in losses[-5:]]}")
print(f"Trend: {' убывает' if losses[-1] < losses[0] else ' растёт или стоит'}")

Step   0 | loss=5.5437
Step   5 | loss=8.3115
Step  10 | loss=11.1517
Step  15 | loss=13.5566
Step  20 | loss=13.3960
Step  25 | loss=14.5622

First 5 losses: ['5.544', '6.658', '7.153', '8.244', '8.152']
Last  5 losses: ['14.562', '13.940', '15.592', '14.129', '16.388']
Trend:  растёт или стоит


## Потюним немного lr с целью повлиять на loss, уменьшить его

In [18]:
code = '''from __future__ import annotations
from typing import Callable
import torch
import torch.nn as nn


class ZeroOrderOptimizer:
    def __init__(
        self,
        model: nn.Module,
        lr: float = 1e-4,
        eps: float = 1e-3,
        perturbation_mode: str = "gaussian",
    ) -> None:
        self.model = model
        self.lr = lr
        self.eps = eps
        self.perturbation_mode = perturbation_mode
        self.step_count = 0
        self.layer_names: list[str] = ["fc.weight", "fc.bias"]

    def _active_params(self) -> dict[str, nn.Parameter]:
        named = dict(self.model.named_parameters())
        missing = [n for n in self.layer_names if n not in named]
        if missing:
            raise KeyError(f"Layer names not found: {missing}")
        return {n: named[n] for n in self.layer_names}

    def _estimate_grad(
        self,
        loss_fn: Callable[[], float],
        params: dict[str, nn.Parameter],
    ) -> dict[str, torch.Tensor]:
        perturbations = {}
        with torch.no_grad():
            for name, param in params.items():
                delta = torch.randint(0, 2, param.shape,
                                      device=param.device).float() * 2 - 1
                perturbations[name] = delta

            for name, param in params.items():
                param.data.add_(self.eps * perturbations[name])
            f_plus = loss_fn()

            for name, param in params.items():
                param.data.sub_(2.0 * self.eps * perturbations[name])
            f_minus = loss_fn()

            for name, param in params.items():
                param.data.add_(self.eps * perturbations[name])

        grad_scalar = (f_plus - f_minus) / (2.0 * self.eps)
        return {name: grad_scalar * perturbations[name] for name in params}

    def _update_params(
        self,
        params: dict[str, nn.Parameter],
        grads: dict[str, torch.Tensor],
    ) -> None:
        with torch.no_grad():
            for name, param in params.items():
                param.data.sub_(self.lr * grads[name])

    def step(self, loss_fn: Callable[[], float]) -> float:
        params = self._active_params()
        with torch.no_grad():
            loss_before = loss_fn()
        grads = self._estimate_grad(loss_fn, params)
        self._update_params(params, grads)
        return float(loss_before)
'''
with open('zo_optimizer.py', 'w') as f:
    f.write(code)
print("OK")

OK


In [19]:
import importlib, sys

# Перезагружаем модуль
for mod in ['zo_optimizer', 'train_data', 'augmentation', 'model', 'head_init']:
    if mod in sys.modules:
        del sys.modules[mod]

from model import get_model
from train_data import get_train_dataset_loader
from zo_optimizer import ZeroOrderOptimizer
import torch.nn as nn

device = torch.device('cuda')
model = get_model().to(device)
optimizer = ZeroOrderOptimizer(model)
_, train_loader = get_train_dataset_loader('/kaggle/working/data', 64)
criterion = nn.CrossEntropyLoss()

def infinite(loader):
    while True:
        yield from loader

data_iter = infinite(train_loader)
losses = []

for i in range(30):
    images, labels = next(data_iter)
    images, labels = images.to(device), labels.to(device)
    def loss_fn(img=images, lbl=labels):
        model.eval()
        with torch.no_grad():
            return float(criterion(model(img), lbl).item())
    losses.append(optimizer.step(loss_fn))
    if i % 5 == 0:
        print(f"Step {i:3d} | loss={losses[-1]:.4f}")

print(f"\nFirst 5: {[f'{x:.3f}' for x in losses[:5]]}")
print(f"Last  5: {[f'{x:.3f}' for x in losses[-5:]]}")
print(f"Trend: {' убывает' if losses[-1] < losses[0] else 'растёт'}")

Step   0 | loss=5.8149
Step   5 | loss=5.7727
Step  10 | loss=5.8492
Step  15 | loss=5.8214
Step  20 | loss=5.5917
Step  25 | loss=5.8586

First 5: ['5.815', '5.680', '5.708', '5.703', '5.850']
Last  5: ['5.859', '5.599', '5.761', '5.893', '5.680']
Trend:  убывает


## Теперь лосс не растет, посчитаем итоговую оценку 

In [20]:
!python validate.py --data_dir /kaggle/working/data --batch_size 32 --n_batches 256 --output results.json

[Device] Using: cuda
[Data] Loading CIFAR100 from '/kaggle/working/data' ...
[Data] Train: 50,000 samples | Val: 10,000 samples

[Checkpoint 1/3] Baseline (ImageNet head)
  Top-1: 0.37%                                                                  

[Checkpoint 2/3] Initialized head (no fine-tuning)
  Top-1: 1.21%                                                                  

[Checkpoint 3/3] ZO fine-tuning (256 steps)
  Active layers: ['fc.weight', 'fc.bias']
  Fine-tuning: 100%|███████████| 256/256 [00:22<00:00, 11.20step/s, loss=5.2031]
  Evaluating fine-tuned model ...
  Top-1: 1.30%                                                                  

 Evaluation Summary
  Checkpoint                        Top-1
------------------------------------------------------------
  1. Baseline (ImageNet head)       0.37%
  2. Initialized head (no FT)       1.21%
  3. Fine-tuned (ZO)                1.30%
------------------------------------------------------------
  Budget: 256 steps ×

## Оказалось все еще низко, воспользуемся технлогией SPSA 

## Алгоритм SPSA : 
### *SPSA (Simultaneous Perturbation Stochastic Approximation) — главная идея проекта.*

### Как работает SPSA — шаг за шагом
1. Генерируем случайный вектор Δ из +1 и -1 (Rademacher) той же формы что и веса
* Δ = random {+1, -1} одного размера с W
2. Делаем два вызова модели — сдвинув веса вперёд и назад
* f⁺ = loss(W + ε·Δ) f⁻ = loss(W - ε·Δ)
3. Считаем псевдо-градиент — насколько и в каком направлении изменился loss
* grad ≈ (f⁺ - f⁻) / (2·ε) × Δ
4. Делаем шаг в сторону уменьшения loss

* W <- W - lr · grad

In [30]:
import torch
import torch.nn as nn
import numpy as np
import sys
sys.path.insert(0, '/kaggle/working/SMILES-2026-ZO-Limited-Resnet/SMILES-2026-ZO-Limited-Resnet')

for mod in list(sys.modules.keys()):
    if mod in ['zo_optimizer','train_data','augmentation','model','head_init']:
        del sys.modules[mod]

from model import get_model
from train_data import get_train_dataset_loader
from augmentation import get_transforms
from torch.utils.data import DataLoader
import torchvision.datasets as datasets

device = torch.device('cuda')
model = get_model().to(device)

# Backbone без головы
backbone = nn.Sequential(*list(model.children())[:-1])
backbone.eval()

_, train_loader = get_train_dataset_loader('/kaggle/working/data', 512)

all_feats, all_labels = [], []
with torch.no_grad():
    for imgs, lbls in train_loader:
        feats = backbone(imgs.to(device)).squeeze(-1).squeeze(-1)
        all_feats.append(feats.cpu())
        all_labels.append(lbls)

X = torch.cat(all_feats)   # (50000, 512)
y = torch.cat(all_labels)  # (50000,)
print(f"X shape: {X.shape}, y shape: {y.shape}")

# One-hot encoding
n_classes = 100
Y = torch.zeros(len(y), n_classes)
Y[torch.arange(len(y)), y] = 1.0  # (50000, 100)


result = torch.linalg.lstsq(X, Y)
W = result.solution  # (512, 100)

print(f"W shape: {W.shape}")

# Транспонируем для fc: fc.weight имеет форму (100, 512)
W_fc = W.T  # (100, 512)
b_fc = torch.zeros(100)

# Сохраняем
torch.save({'weight': W_fc, 'bias': b_fc}, '/kaggle/working/pinv_head.pt')
print("Saved pinv_head.pt ")

# Быстрая проверка точности
with torch.no_grad():
    model.fc.weight.copy_(W_fc)
    model.fc.bias.copy_(b_fc)

model.eval()
val_dataset = datasets.CIFAR100(
    root='/kaggle/working/data', train=False,
    download=False, transform=get_transforms(train=False)
)
val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False)

correct, total = 0, 0
with torch.no_grad():
    for imgs, lbls in val_loader:
        out = model(imgs.to(device))
        correct += (out.argmax(1) == lbls.to(device)).sum().item()
        total += lbls.size(0)
print(f"Val accuracy (pinv head, no ZO): {correct/total*100:.2f}%")

  [head_init] Loaded pseudo-inverse weights 
X shape: torch.Size([50000, 512]), y shape: torch.Size([50000])
W shape: torch.Size([512, 100])
Saved pinv_head.pt 
Val accuracy (pinv head, no ZO): 59.32%


## Изменим head_init.py

In [29]:
code = '''import torch
import torch.nn as nn
import os


def init_last_layer(layer: nn.Linear) -> None:
    path = "/kaggle/working/pinv_head.pt"
    if os.path.exists(path):
        weights = torch.load(path, map_location="cpu")
        with torch.no_grad():
            layer.weight.copy_(weights["weight"])
            layer.bias.copy_(weights["bias"])
        print("  [head_init] Loaded pseudo-inverse weights ")
    else:
        nn.init.xavier_uniform_(layer.weight)
        nn.init.zeros_(layer.bias)
        print("  [head_init] Xavier init (fallback)")
'''
with open('head_init.py', 'w') as f:
    f.write(code)
print("head_init.py updated ")

head_init.py updated 


In [23]:
!python validate.py --data_dir /kaggle/working/data --batch_size 32 --n_batches 256 --output results.json

[Device] Using: cuda
[Data] Loading CIFAR100 from '/kaggle/working/data' ...
[Data] Train: 50,000 samples | Val: 10,000 samples

[Checkpoint 1/3] Baseline (ImageNet head)
  Top-1: 0.37%                                                                  

[Checkpoint 2/3] Initialized head (no fine-tuning)
  [head_init] Loaded pseudo-inverse weights ✓
  Top-1: 59.53%                                                                 

[Checkpoint 3/3] ZO fine-tuning (256 steps)
  Active layers: ['fc.weight', 'fc.bias']
  Fine-tuning: 100%|███████████| 256/256 [00:23<00:00, 11.11step/s, loss=4.3720]
  Evaluating fine-tuned model ...
  Top-1: 23.93%                                                                 

 Evaluation Summary
  Checkpoint                        Top-1
------------------------------------------------------------
  1. Baseline (ImageNet head)       0.37%
  2. Initialized head (no FT)      59.53%
  3. Fine-tuned (ZO)               23.93%
------------------------------------

## Результат все еще не впечатляющий 

#### ZO-оптимизация сломала хорошую инициализацию — из  59.76% стало 23.91%. Это может быть проблема с SPSA с lr=1e-4 слишком грубый для хорошо настроенных весов. Попробуем обновить zo_optimizer.py с маленьким lr

In [24]:
code = '''from __future__ import annotations
from typing import Callable
import torch
import torch.nn as nn


class ZeroOrderOptimizer:
    def __init__(
        self,
        model: nn.Module,
        lr: float = 1e-6,
        eps: float = 1e-3,
        perturbation_mode: str = "gaussian",
    ) -> None:
        self.model = model
        self.lr = lr
        self.eps = eps
        self.perturbation_mode = perturbation_mode
        self.step_count = 0
        self.layer_names: list[str] = ["fc.weight", "fc.bias"]

    def _active_params(self) -> dict[str, nn.Parameter]:
        named = dict(self.model.named_parameters())
        missing = [n for n in self.layer_names if n not in named]
        if missing:
            raise KeyError(f"Layer names not found: {missing}")
        return {n: named[n] for n in self.layer_names}

    def _estimate_grad(
        self,
        loss_fn: Callable[[], float],
        params: dict[str, nn.Parameter],
    ) -> dict[str, torch.Tensor]:
        perturbations = {}
        with torch.no_grad():
            for name, param in params.items():
                delta = torch.randint(0, 2, param.shape,
                                      device=param.device).float() * 2 - 1
                perturbations[name] = delta

            for name, param in params.items():
                param.data.add_(self.eps * perturbations[name])
            f_plus = loss_fn()

            for name, param in params.items():
                param.data.sub_(2.0 * self.eps * perturbations[name])
            f_minus = loss_fn()

            for name, param in params.items():
                param.data.add_(self.eps * perturbations[name])

        grad_scalar = (f_plus - f_minus) / (2.0 * self.eps)
        return {name: grad_scalar * perturbations[name] for name in params}

    def _update_params(
        self,
        params: dict[str, nn.Parameter],
        grads: dict[str, torch.Tensor],
    ) -> None:
        with torch.no_grad():
            for name, param in params.items():
                param.data.sub_(self.lr * grads[name])

    def step(self, loss_fn: Callable[[], float]) -> float:
        params = self._active_params()
        with torch.no_grad():
            loss_before = loss_fn()
        grads = self._estimate_grad(loss_fn, params)
        self._update_params(params, grads)
        return float(loss_before)
'''
with open('zo_optimizer.py', 'w') as f:
    f.write(code)
print("OK")

OK


### Посмотрим как ведут себя лосс перед запуском на полном обучении выборки

In [25]:
import torch, torch.nn as nn, sys
for mod in list(sys.modules.keys()):
    if mod in ['zo_optimizer','train_data','augmentation','model','head_init']:
        del sys.modules[mod]

from model import get_model
from train_data import get_train_dataset_loader
from zo_optimizer import ZeroOrderOptimizer

device = torch.device('cuda')
model = get_model().to(device)

# Загружаем pinv веса
weights = torch.load('/kaggle/working/pinv_head.pt')
with torch.no_grad():
    model.fc.weight.copy_(weights['weight'])
    model.fc.bias.copy_(weights['bias'])

optimizer = ZeroOrderOptimizer(model)
_, train_loader = get_train_dataset_loader('/kaggle/working/data', 64)
criterion = nn.CrossEntropyLoss()

def infinite(loader):
    while True:
        yield from loader

data_iter = infinite(train_loader)
losses = []

for i in range(20):
    imgs, lbls = next(data_iter)
    imgs, lbls = imgs.to(device), lbls.to(device)
    def loss_fn(im=imgs, lb=lbls):
        model.eval()
        with torch.no_grad():
            return float(criterion(model(im), lb).item())
    losses.append(optimizer.step(loss_fn))

print(f"First 5: {[f'{x:.4f}' for x in losses[:5]]}")
print(f"Last  5: {[f'{x:.4f}' for x in losses[-5:]]}")
print(f"Trend: {'убывает' if losses[-1] < losses[0] else 'растёт'}")

  [head_init] Loaded pseudo-inverse weights ✓
First 5: ['4.4018', '4.3768', '4.4371', '4.4071', '4.3962']
Last  5: ['4.3935', '4.4275', '4.4296', '4.4348', '4.4039']
Trend: растёт


#### Loss практически не меняется (4.41 +- 0.01) — это хороший знак, модель не ломается. Просто тренд чуть вверх из-за шума SPSA, а не реального роста. Попробуем добавить early stopping: если шаг ухудшает loss — откатываем его назад.

In [26]:
code = '''from __future__ import annotations
from typing import Callable
import torch
import torch.nn as nn


class ZeroOrderOptimizer:
    def __init__(
        self,
        model: nn.Module,
        lr: float = 1e-6,
        eps: float = 1e-3,
        perturbation_mode: str = "gaussian",
    ) -> None:
        self.model = model
        self.lr = lr
        self.eps = eps
        self.perturbation_mode = perturbation_mode
        self.step_count = 0
        self.layer_names: list[str] = ["fc.weight", "fc.bias"]

    def _active_params(self) -> dict[str, nn.Parameter]:
        named = dict(self.model.named_parameters())
        missing = [n for n in self.layer_names if n not in named]
        if missing:
            raise KeyError(f"Layer names not found: {missing}")
        return {n: named[n] for n in self.layer_names}

    def _estimate_grad(
        self,
        loss_fn: Callable[[], float],
        params: dict[str, nn.Parameter],
    ) -> dict[str, torch.Tensor]:
        perturbations = {}
        with torch.no_grad():
            for name, param in params.items():
                delta = torch.randint(0, 2, param.shape,
                                      device=param.device).float() * 2 - 1
                perturbations[name] = delta

            for name, param in params.items():
                param.data.add_(self.eps * perturbations[name])
            f_plus = loss_fn()

            for name, param in params.items():
                param.data.sub_(2.0 * self.eps * perturbations[name])
            f_minus = loss_fn()

            for name, param in params.items():
                param.data.add_(self.eps * perturbations[name])

        grad_scalar = (f_plus - f_minus) / (2.0 * self.eps)
        return {name: grad_scalar * perturbations[name] for name in params}

    def _update_params(
        self,
        params: dict[str, nn.Parameter],
        grads: dict[str, torch.Tensor],
    ) -> None:
        with torch.no_grad():
            for name, param in params.items():
                param.data.sub_(self.lr * grads[name])

    def step(self, loss_fn: Callable[[], float]) -> float:
        params = self._active_params()

        with torch.no_grad():
            loss_before = loss_fn()

        # Сохраняем копии весов перед шагом
        backup = {name: param.data.clone() for name, param in params.items()}

        grads = self._estimate_grad(loss_fn, params)
        self._update_params(params, grads)

        # Проверяем — если стало хуже, откатываем
        with torch.no_grad():
            loss_after = loss_fn()

        if loss_after > loss_before:
            with torch.no_grad():
                for name, param in params.items():
                    param.data.copy_(backup[name])

        return float(loss_before)
'''
with open('zo_optimizer.py', 'w') as f:
    f.write(code)
print("OK")

OK


In [27]:
!python validate.py --data_dir /kaggle/working/data --batch_size 32 --n_batches 256 --output results.json

[Device] Using: cuda
[Data] Loading CIFAR100 from '/kaggle/working/data' ...
[Data] Train: 50,000 samples | Val: 10,000 samples

[Checkpoint 1/3] Baseline (ImageNet head)
  Top-1: 0.37%                                                                  

[Checkpoint 2/3] Initialized head (no fine-tuning)
  [head_init] Loaded pseudo-inverse weights ✓
  Top-1: 59.53%                                                                 

[Checkpoint 3/3] ZO fine-tuning (256 steps)
  Active layers: ['fc.weight', 'fc.bias']
  Fine-tuning: 100%|███████████| 256/256 [00:30<00:00,  8.52step/s, loss=4.3844]
  Evaluating fine-tuned model ...
  Top-1: 59.68%                                                                 

 Evaluation Summary
  Checkpoint                        Top-1
------------------------------------------------------------
  1. Baseline (ImageNet head)       0.37%
  2. Initialized head (no FT)      59.53%
  3. Fine-tuned (ZO)               59.68%
------------------------------------

## Выводы:
 * ### достигли достаточного неплохо качетсва без использования градиентов и грандиентного спуска в примерно ~60% 

In [33]:
# import os
# os.chdir('/kaggle/working/SMILES-2026-ZO-Limited-Resnet')

# print("Доступные файлы:")
# !ls -la

# !zip -r /kaggle/working/smiles_complete_solution.zip \
#     results.json \
#     SOLUTION.md \
#     zo_optimizer.py \
#     head_init.py \
#     augmentation.py \
#     train_data.py \
#     model.py \
#     validate.py

# print("Архив создан: /kaggle/working/smiles_complete_solution.zip")

Доступные файлы:
total 72
drwxr-xr-x 4 root root  4096 May  2 14:18 .
drwxr-xr-x 5 root root  4096 May  2 14:14 ..
-rw-r--r-- 1 root root   694 May  2 14:04 augmentation.py
drwxr-xr-x 8 root root  4096 May  2 14:04 .git
-rw-r--r-- 1 root root  3888 May  2 14:04 .gitignore
-rw-r--r-- 1 root root   553 May  2 14:18 head_init.py
-rw-r--r-- 1 root root  1580 May  2 14:04 model.py
drwxr-xr-x 2 root root  4096 May  2 14:18 __pycache__
-rw-r--r-- 1 root root  7407 May  2 14:04 README.md
-rw-r--r-- 1 root root    47 May  2 14:04 requirements.txt
-rw-r--r-- 1 root root   251 May  2 14:18 results.json
-rw-r--r-- 1 root root  3163 May  2 14:18 SOLUTION.md
-rw-r--r-- 1 root root  1188 May  2 14:04 train_data.py
-rw-r--r-- 1 root root 11719 May  2 14:04 validate.py
-rw-r--r-- 1 root root  2840 May  2 14:16 zo_optimizer.py
  adding: results.json (deflated 41%)
  adding: SOLUTION.md (deflated 47%)
  adding: zo_optimizer.py (deflated 67%)
  adding: head_init.py (deflated 49%)
  adding: augmentation.py